# D2 · Calibración espectral

**Spec:** [`docs/spec_D2_codex_spectral_calibration.md`](../docs/spec_D2_codex_spectral_calibration.md)  |  **Bloque:** D · Método  |  **Run de este set:** `ROXs42Bb_realigned`

Calibra los 6 métodos y la primaria (Δλ, escala de flujo, continuo, presupuesto de error) y produce los espectros definitivos, con `spec_final_object.fits` como canónico.

| | |
|---|---|
| **Entrada** | Los 6 métodos + la primaria de C4 + M3 |
| **Salida (QC/productos)** | `stages/stage_x11_qc.json`, `spec_final_object.fits`, `spec_calibrated_<método>_object.fits` (6), `spec_calibrated_psffit_star.fits` |
| **Consume aguas abajo** | E1, E3, G2 |


## Qué hace D2 y qué integramos

**Los espectros definitivos son el entregable de esta etapa**, no solo la entrada de E1: el compañero por los **6 métodos** (misma rejilla, superponibles) y la **primaria** (`spec_calibrated_psffit_star.fits`), todos con unidad declarada (`BUNIT`) y error total. El canónico se copia además a `spec_final_object.fits`.

D2 convierte el espectro canónico (psffit) en el **producto científico final**: λ corregida y en marco declarado, flujo en escala validada, continuo por dos vías, y un **error total con presupuesto de sistemáticos explícito**. **Aplica factores medidos aguas arriba** (trazables al QC que los midió) — no mide nada nuevo.

- **λ:** Δλ = −0.052 Å (de A4/M1), marco final **baricéntrico**.
- **Flujo:** escala = 1.0 (M3 factor 0.957 vs Gaia DR3, estado **green**: se aplica escala 1.0 porque el factor es consistente con 1 dentro de su error).
- **Continuo:** running-median y polinomio; su diferencia es el término `sys_continuum`.
- **Error:** `stat` (empírico, M5 rojo) + sistemáticos (flujo-cal, psf, cielo, telúrico, continuo). El total está **dominado por el stat**.

**Diagnóstico del 'continuo rojo inestable'** ([`docs/d2_red_continuum_diagnosis.md`](../docs/d2_red_continuum_diagnosis.md)): son 3 cosas reales (señal de enana fría + sistemático de nivel inter-método + rigidez del polinomio), **no** un defecto de PSF.

**La referenciación a controles que integramos (2026-07-11):** `v3_continuum_stable` gatea ahora sobre la concordancia inter-método **control-referenciada** (**0.867**, consistente con el t control-centrado de D1), guardando la cruda (0.317) como diagnóstico; el sistemático rojo baja de **1.76× → 1.35×**; y el producto final lleva la columna `cont_runmed_biasref` para G3. v3 sigue <0.90 → falla por el sistemático cromático **genuino**, no por el pedestal. **No afecta la línea Hα ni el límite de Ṁ.**


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs42Bb_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage_x11_calibrate.sh --run-id $RUN
```

Ligero–moderado.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_x11_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage_x11_calibrate.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_x11_qc.json', RUN_ID)
nb.show(qc, keys=['canonical_method', 'scale_factor', 'v3_continuum_stable.ok', 'fraction_channels_methods_agree', 'control_referenced'], title='D2')


## Resultados que llevaron a la conclusión

Calibración aplicada + la referenciación de continuo (antes/después) del `stage_x11_qc.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('D2', 'stages/stage_x11_qc.json'):
        q = nb.load_qc('stages/stage_x11_qc.json', RUN_ID)
        print('canónico:', q['canonical_method'])
        print(f"λ: Δλ={q['wavelength']['dlambda_A']:.3f} Å, marco={q['wavelength']['frame_final']}, {q['wavelength']['status']}")
        print(f"flujo: escala={q['flux']['scale_factor']:.3f} ({q['flux']['source'][:60]}...)")
        if q['flux'].get('declared_err_frac'):
            print(f"       flujo-cal declarado (NO en el total): {100 * q['flux']['declared_err_frac']:.1f}%"
                  f" — {q['flux']['declared_source'][:70]}")
        # La unidad que resolverán E3/G3: knob -> BUNIT del producto -> QC de M3.
        un = q['flux'].get('unit')
        if un:
            print(f"       unidad: {un['cgs']} erg/s/cm²/Å vía {un['source']}"
                  f" (BUNIT={un['from_bunit']}, M3={un['from_m3_qc']})")
            if un['conflict']:
                print('       ⚠ ' + un['conflict'])
        im = q['continuum']['intermethod_systematic']; ar = im['after_control_reference']
        print()
        print('continuo inter-método (psffit vs optimal_psfsub):')
        print(f"  ANTES  (crudo)       fraction_agree={im['fraction_channels_methods_agree']:.3f}  red_ratio={im['red_band_median_ratio']:.2f}×")
        print(f"  DESPUÉS (referenciado) fraction_agree={ar['fraction_channels_methods_agree']:.3f}  red_ratio={ar['red_band_median_ratio']:.2f}×")
        print(f"  sesgo rojo psffit/psfsub = {ar['canonical_control_bias_red']:+.0f} / {ar['other_control_bias_red']:+.0f}")
        v3 = q['checks']['v3_continuum_stable']
        print(f"\nv3_continuum_stable: ok={v3['ok']} (métrica={v3['metric']}, umbral {v3['threshold']}) -> falla por el sistemático genuino")

        # Los espectros definitivos: la tabla que D2 deja en su QC.
        sp = q.get('spectra')
        if sp is None:
            print('\n(este QC es anterior a la tabla de espectros: re-ejecuta D2 para tenerla)')
        else:
            n = lambda v, f='{:.1f}': '—' if v is None else f.format(v)
            print(f"\nespectros definitivos: {sp['n_companion']} métodos + {sp['n_primary']} primaria"
                  f" | unidad {sp['unit']} (consistente={sp['unit_consistent']})"
                  f" | misma rejilla={sp['companions_share_grid']}")
            print(f"medianas en {sp['red_band_A'][0]:.0f}–{sp['red_band_A'][1]:.0f} Å (donde el compañero se detecta)")
            print(f"  {'espectro':16s} {'papel':10s} {'S/N':>7s} {'flujo':>10s} {'err total':>10s} {'ratio/canon':>12s}")
            for row in sp['table']:
                tag = row['name'] + (' *' if row['canonical'] else '')
                print(f"  {tag:16s} {row['role']:10s} {n(row['red_snr_median']):>7s}"
                      f" {n(row['red_flux_median']):>10s} {n(row['flux_err_total_median']):>10s}"
                      f" {n(row['ratio_to_canonical_red'], '{:.2f}×'):>12s}")
            for row in sp['table']:
                if row['caveat']:
                    print(f"  ! {row['name']}: {row['caveat']}")


## Plot 1 — la referenciación a controles (antes/después)

Continuos de los dos métodos G1-validados (psffit, optimal_psfsub), crudos vs referenciados a sus controles. En el **rojo** (>7500 Å, donde el compañero se detecta) los referenciados **concuerdan**; la concordancia global sube de 0.317 a 0.867. *(El azul, λ<7000, tiene SNR<1: su discrepancia está dentro del error combinado — no es señal.)*


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    rd = nb.run_dir(RUN_ID)
    qx = nb.load_qc('stages/stage_x11_qc.json', RUN_ID)
    im = qx['continuum']['intermethod_systematic']; ar = im['after_control_reference']
    def col(fn, c):
        h = fits.open(rd / 'stages' / fn); v = np.asarray(h[1].data[c], float); h.close(); return v
    wave = col('spec_calibrated_psffit_object.fits', 'wave_A')   # eje λ del producto
    fig, (axl, axr) = plt.subplots(1, 2, figsize=(13, 4.3), sharey=True)
    axl.plot(wave, col('spec_calibrated_psffit_object.fits', 'cont_runmed'), lw=1.1, color='tab:blue', label='psffit')
    axl.plot(wave, col('spec_calibrated_optimal_psfsub_object.fits', 'cont_runmed'), lw=1.1, color='tab:orange', label='optimal_psfsub')
    axl.set_title(f"ANTES: continuos crudos (concuerdan {im['fraction_channels_methods_agree']:.3f})")
    axr.plot(wave, col('spec_calibrated_psffit_object.fits', 'cont_runmed_biasref'), lw=1.1, color='tab:blue', label='psffit ref')
    axr.plot(wave, col('spec_calibrated_optimal_psfsub_object.fits', 'cont_runmed_biasref'), lw=1.1, color='tab:orange', label='optimal_psfsub ref')
    axr.set_title(f"DESPUÉS: referenciados a controles ({ar['fraction_channels_methods_agree']:.3f})")
    for ax in (axl, axr):
        ax.set_xlabel('λ [Å]'); ax.axvline(6563, color='tab:red', ls=':'); ax.axhline(0, color='0.7', lw=0.6); ax.legend(fontsize=8)
    axl.set_ylabel('continuo'); axl.set_ylim(-1500, 1500)
    axr.text(5000, -1300, 'azul: SNR<1\n(dentro del error)', fontsize=7, color='0.4')
    fig.suptitle('D2 · referenciación a controles: acerca los dos métodos G1-validados')
    fig.tight_layout()
    outdir = rd / 'plots' / 'd2_calibrate'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'referencing.png', dpi=110); print('figura ->', outdir / 'referencing.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — el presupuesto de error

Cada componente del error del `spec_final_object.fits` vs λ (escala log). El total está **dominado por el `stat`** (empírico, M5 rojo); el sistemático de continuo (runmed vs poly) es el segundo; flujo-cal/cielo/telúrico son ~0.

La línea **discontinua** es `sys_fluxcal_declared`: el desvío de calibración absoluta que M3 mide frente a Gaia (|1−`flux_factor`|), **declarado y no sumado** a `flux_err_total` — M3 no publica barra de error, y plegarlo movería el error del compañero, que sostiene decisiones congeladas. G3 ya asume su propio 10% (`g3_sys_fluxcal_frac`), mayor que este valor.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    rd = nb.run_dir(RUN_ID)
    h = fits.open(rd / 'stages' / 'spec_final_object.fits'); d = h[1].data
    wave = np.asarray(d['wave_A'], float)   # eje λ del propio producto
    comp = {'stat': 'flux_err_stat', 'flujo-cal': 'sys_fluxcal', 'psf': 'sys_psf',
            'cielo': 'sys_sky', 'telúrico': 'sys_telluric', 'continuo': 'sys_continuum'}
    sm = lambda x, n=51: np.convolve(np.nan_to_num(np.abs(x)), np.ones(n) / n, mode='same')
    fig, ax = plt.subplots(figsize=(11, 4))
    for lab, c in comp.items():
        if c in d.columns.names:
            ax.plot(wave, sm(np.asarray(d[c], float)), lw=1, label=lab)
    ax.plot(wave, sm(np.asarray(d['flux_err_total'], float)), lw=2, color='k', label='TOTAL')
    if 'sys_fluxcal_declared' in d.columns.names:   # declarado, NO sumado
        ax.plot(wave, sm(np.asarray(d['sys_fluxcal_declared'], float)), lw=1.3,
                ls='--', color='tab:pink', label='flujo-cal DECLARADO (no en el total)')
    h.close()
    ax.set_yscale('log'); ax.set_xlabel('λ [Å]'); ax.set_ylabel('error (|componente|, suavizado)')
    ax.set_title('D2 · presupuesto de error: stat + sistemáticos'); ax.legend(fontsize=8, ncol=4)
    outdir = rd / 'plots' / 'd2_calibrate'; outdir.mkdir(parents=True, exist_ok=True)
    fig.tight_layout(); fig.savefig(outdir / 'error_budget.png', dpi=110)
    print('figura ->', outdir / 'error_budget.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 3 — los espectros definitivos (6 métodos + la primaria)

El entregable de D2 en una figura, dibujada con la **misma función que usa la etapa** (`musepipe.stages.definitive_spectra_figure`), leyendo los productos calibrados del run:

- **arriba:** la **primaria** con su banda de error total (stat + sistemáticos). Está ~1e3–1e4 veces por encima del compañero, así que necesita su propio panel.
- **centro:** el **compañero por los 6 métodos**, suavizado 15 canales para que se lean a la vez, sobre la banda de error total del canónico (sin suavizar). Hα en rojo punteado.
- **abajo:** el **continuo referenciado a controles** — la comparación inter-método real (la del gate v3); la banda sombreada es el rojo, donde el compañero se detecta.

*(`sgf` filtra el continuo por construcción: su nivel no es comparable, su valor está en la línea.)*


In [ ]:
try:
    import matplotlib.pyplot as plt
    from musepipe.stages import definitive_spectra_figure
    rd = nb.run_dir(RUN_ID)
    fig, axes = definitive_spectra_figure(rd / 'stages', plt=plt)
    outdir = rd / 'plots' / 'd2_calibrate'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'definitive_spectra.png', dpi=110)
    print('figura ->', outdir / 'definitive_spectra.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **Los espectros definitivos (6 métodos + primaria) son el entregable de D2**, antes del estudio de Hα: todos con `BUNIT` declarado y error total; la tabla vive en `qc['spectra']` y la figura en `plots/stage_x11_spectra.png`. · [`docs/espectros_definitivos_handoff_2026-07-25.md`](../docs/espectros_definitivos_handoff_2026-07-25.md)
- **Diagnóstico honesto del 'continuo rojo inestable'**: 3 cosas reales (señal de enana fría + sistemático de nivel inter-método + rigidez del polinomio). NO es defecto de PSF. · [`docs/d2_red_continuum_diagnosis.md`](../docs/d2_red_continuum_diagnosis.md)
- **Referenciación a controles integrada** (2026-07-11): v3 gatea sobre la métrica referenciada (0.867 vs cruda 0.317); sistemático rojo 1.76×→1.35×; columna `cont_runmed_biasref` entregada para G3.
- D1 **ya era control-centrado** (su veredicto refleja el sistemático genuino); esto solo puso a D2 al mismo nivel. v3 sigue <0.90 → sistemático cromático genuino, limitación aceptada en F1.
- El sistemático rojo NO afecta la línea Hα ni el límite de Ṁ; escala de flujo 1.0 validada vs Gaia; error total dominado por el stat.


## Conclusión (registrada)

**D2: producto final `spec_final_object` (psffit); Δλ −0.074 Å baricéntrico; flujo escala 1.0 (Gaia); error dominado por el stat.**

- **Fecha:** calibración D1 v2 realineado (2026-07-09); referenciación integrada 2026-07-11.
- **λ/flujo:** todos los factores trazables a A4 (M1 offset, M3 Gaia).
- **Continuo:** referenciación a controles integrada — v3 sobre 0.867 (cruda 0.317), rojo 1.76×→1.35×, columna `cont_runmed_biasref` para G3.
- **v3 sigue fallando** (0.867<0.90) por el sistemático cromático genuino → limitación aceptada; F1 sigue yellow.
- **Endpoint intacto:** la línea Hα (E1) y el límite de Ṁ (E3) no dependen de esto.
- **Espectros definitivos:** los 6 métodos + la primaria, con unidad y error total (tabla en `qc['spectra']`, figura en `plots/stage_x11_spectra.png`) — resultado en sí mismos, entregados antes del estudio de Hα.
- **Unidad de flujo:** `qc['flux']['unit']` deja resuelta y contrastada la escala que usarán E3/G3 (knob → `BUNIT` → `m3_flux.flux_unit_cgs`). Si el `BUNIT` del producto y la unidad con la que M3 midió el factor no coinciden, sale como `open_issue`.
- **Downstream:** `spec_final_object` alimenta E1, E3 y G2.
